# Sensitivity analysis

## 1. Imports and visualization settings

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import logging
import sys
import pickle
from pathlib import Path
from src import (
    perturb_picks_gaussian,
    perturb_picks_bias,
    compute_sensitivity_metrics,
    compute_monte_carlo_statistics,
    create_sensitivity_summary,
    save_intermediate_results,
    set_plot_style,
    analyze_all_windows,
    segment_all_signals
)

colors, colors1 = set_plot_style()

logging.basicConfig(level=logging.INFO, format="%(levelname)s | %(message)s")
logger = logging.getLogger()

def check(condition, message):
    if condition:
        logger.info(message)
    else:
        raise ValueError(message)

logger.info("Environment ready")

INFO | Environment ready


## Configuration


In [2]:
# Dataset selection
DATASET = 'current'  # Options: 'current', 'aquila'
PICKING_METHOD = 'phasenet'  # Options: 'ar_pick', 'phasenet'

# Data types and methods to analyze
DATA_TYPES = ['acceleration', 'velocity', 'displacement']  # Can select subset
CODA_METHODS = ['rautian', 'arias', 'envelope', 'median']  # Can select subset

# Current analysis (single run for development/testing)
DATA_TYPE = 'acceleration'  # Current data type being analyzed
CODA_METHOD = 'rautian'  # Current coda method being analyzed

# Sensitivity analysis parameters
PERTURBATION_SCENARIOS = {
    'noise_small': {'type': 'gaussian', 'std': 0.2},
    'noise_medium': {'type': 'gaussian', 'std': 0.5},
    'noise_large': {'type': 'gaussian', 'std': 1.0},
    'bias_early': {'type': 'bias', 'bias': -0.5},
    'bias_late': {'type': 'bias', 'bias': 0.5}
}

# Monte Carlo parameters
N_MONTE_CARLO = 100  # Number of Monte Carlo runs
MONTE_CARLO_STD = 0.5  # Standard deviation for Monte Carlo (seconds)

# Moment scaling parameters (same as notebook 04a)
TAU_MIN = 0.01  # seconds
Q_VALUES = np.linspace(0.25, 5.0, 20)
SAMPLING_RATE = 200.0  # Hz

# Signal column mapping
SIGNAL_COLUMN_MAP = {
    'acceleration': 'signal',
    'velocity': 'signal',
    'displacement': 'signal'
}

SIGNAL_UNIT_MAP = {
    'acceleration': 'cm/s²',
    'velocity': 'cm/s',
    'displacement': 'cm'
}

PEAK_COLUMN_MAP = {
    'acceleration': 'PGA_CM/S^2',
    'velocity': 'PGV_CM/S',
    'displacement': 'PGD_CM'
}

TIME_PEAK_COLUMN_MAP = {
    'acceleration': 'TIME_PGA_S',
    'velocity': 'TIME_PGV_S',
    'displacement': 'TIME_PGD_S'
}

# Set current values
SIGNAL_COLUMN = SIGNAL_COLUMN_MAP[DATA_TYPE]
SIGNAL_UNIT = SIGNAL_UNIT_MAP[DATA_TYPE]
PEAK_COLUMN = PEAK_COLUMN_MAP[DATA_TYPE]
TIME_PEAK_COLUMN = TIME_PEAK_COLUMN_MAP[DATA_TYPE]

logger.info(f"Dataset: {DATASET}")
logger.info(f"Picking method: {PICKING_METHOD}")
logger.info(f"Current analysis: {DATA_TYPE} / {CODA_METHOD}")
logger.info(f"Perturbation scenarios: {len(PERTURBATION_SCENARIOS)}")
logger.info(f"Monte Carlo runs: {N_MONTE_CARLO}")

INFO | Dataset: current
INFO | Picking method: phasenet
INFO | Current analysis: acceleration / rautian
INFO | Perturbation scenarios: 5
INFO | Monte Carlo runs: 100


## Data loading

In [3]:
# Get project root
NOTEBOOK_DIR = Path.cwd()
PROJECT_ROOT = NOTEBOOK_DIR.parent

# Define input paths (where to load baseline data from)
METADATA_PROCESSED = PROJECT_ROOT / 'data' / 'processed' / '01a_metadata' / DATA_TYPE

if PICKING_METHOD == 'ar_pick':
    PICKS_INPUT_DIR = PROJECT_ROOT / 'data' / 'processed' / '03a_phase_identification_ar_pick' / DATA_TYPE
    BASELINE_RESULTS_DIR = PROJECT_ROOT / 'data' / 'processed' / '04a_moment_scaling_spatial' / 'ar_pick' / DATA_TYPE
    BASELINE_FIGURES_DIR = PROJECT_ROOT / 'figures' / '04a_moment_scaling_spatial' / 'ar_pick' / DATA_TYPE
elif PICKING_METHOD == 'phasenet':
    PICKS_INPUT_DIR = PROJECT_ROOT / 'data' / 'processed' / '03b_phase_identification_phasenet' / DATA_TYPE
    BASELINE_RESULTS_DIR = PROJECT_ROOT / 'data' / 'processed' / '04a_moment_scaling_spatial' / 'phasenet' / DATA_TYPE
    BASELINE_FIGURES_DIR = PROJECT_ROOT / 'figures' / '04a_moment_scaling_spatial' / 'phasenet' / DATA_TYPE

# Define output paths (for sensitivity analysis results)
SENSITIVITY_OUTPUT_DIR = PROJECT_ROOT / 'data' / 'processed' / '05_sensitivity_analysis' / PICKING_METHOD / DATA_TYPE
SENSITIVITY_FIGURES_DIR = PROJECT_ROOT / 'figures' / '05_sensitivity_analysis' / PICKING_METHOD / DATA_TYPE
SENSITIVITY_INTERMEDIATE_DIR = SENSITIVITY_OUTPUT_DIR / 'intermediate'

# Create output directories
SENSITIVITY_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
SENSITIVITY_FIGURES_DIR.mkdir(parents=True, exist_ok=True)
SENSITIVITY_INTERMEDIATE_DIR.mkdir(parents=True, exist_ok=True)

check(SENSITIVITY_OUTPUT_DIR.exists(), f"Sensitivity output directory ready: {SENSITIVITY_OUTPUT_DIR}")
check(SENSITIVITY_FIGURES_DIR.exists(), f"Sensitivity figures directory ready: {SENSITIVITY_FIGURES_DIR}")

# ==============================================================================
# 3. LOAD BASELINE DATA
# ==============================================================================

logger.info("LOADING BASELINE DATA")

# Load signals_dict (raw signals - needed for all perturbations)
logger.info(f"\nLoading signals_dict for {DATA_TYPE}...")
signals_file = PICKS_INPUT_DIR / f'signals_{DATA_TYPE}_dict.pkl'
with open(signals_file, 'rb') as f:
    signals_dict = pickle.load(f)
logger.info(f"Loaded: {signals_file}")
logger.info(f"Stations: {len(signals_dict)}")

# Load df_full (picks and metadata - will be perturbed)
logger.info(f"\nLoading df_full with picks and metadata...")
df_full_file = PICKS_INPUT_DIR / f'df_full_{DATA_TYPE}_{PICKING_METHOD}.parquet'
df_full = pd.read_parquet(df_full_file)
logger.info(f"Loaded: {df_full_file}")
logger.info(f"Shape: {df_full.shape}")

# Load baseline moment scaling results (original ζ for comparison)
logger.info(f"\nLoading baseline moment scaling results...")
baseline_results = {}

for method in CODA_METHODS:
    result_file = BASELINE_RESULTS_DIR / method / 'ensemble_spatial_summary.parquet'
    if result_file.exists():
        baseline_results[method] = pd.read_parquet(result_file)
        logger.info(f"Loaded baseline for {method}: {result_file}")
    else:
        logger.warning(f"Baseline not found for {method}: {result_file}")

logger.info("\nBaseline data loaded successfully")
logger.info(f"Available baseline methods: {list(baseline_results.keys())}")

INFO | Sensitivity output directory ready: /Users/giulianaparadiso/Desktop/PoliTo/Tesi/tesi-seismic-analysis/data/processed/05_sensitivity_analysis/phasenet/acceleration
INFO | Sensitivity figures directory ready: /Users/giulianaparadiso/Desktop/PoliTo/Tesi/tesi-seismic-analysis/figures/05_sensitivity_analysis/phasenet/acceleration
INFO | LOADING BASELINE DATA
INFO | 
Loading signals_dict for acceleration...
INFO | Loaded: /Users/giulianaparadiso/Desktop/PoliTo/Tesi/tesi-seismic-analysis/data/processed/03b_phase_identification_phasenet/acceleration/signals_acceleration_dict.pkl
INFO | Stations: 21
INFO | 
Loading df_full with picks and metadata...
INFO | Loaded: /Users/giulianaparadiso/Desktop/PoliTo/Tesi/tesi-seismic-analysis/data/processed/03b_phase_identification_phasenet/acceleration/df_full_acceleration_phasenet.parquet
INFO | Shape: (63, 85)
INFO | 
Loading baseline moment scaling results...
INFO | Loaded baseline for rautian: /Users/giulianaparadiso/Desktop/PoliTo/Tesi/tesi-seis

## Sensitivity analysis

In [4]:
logger.info("STARTING SENSITIVITY ANALYSIS")

# Initialize results dictionary
sensitivity_results = {
    data_type: {
        method: {
            scenario: {
                'pre_event': None,
                'p_wave': None,
                's_wave': None,
                'coda': None
            } for scenario in list(PERTURBATION_SCENARIOS.keys()) + ['monte_carlo']
        } for method in CODA_METHODS
    } for data_type in DATA_TYPES
}

# Track progress
total_runs = len(DATA_TYPES) * len(CODA_METHODS) * (len(PERTURBATION_SCENARIOS) + 1)
current_run = 0

logger.info(f"Total configurations to test: {total_runs}")
logger.info(f"Data types: {DATA_TYPES}")
logger.info(f"Coda methods: {CODA_METHODS}")
logger.info(f"Scenarios: {list(PERTURBATION_SCENARIOS.keys()) + ['monte_carlo']}")

# ==============================================================================
# Loop over data types
# ==============================================================================

for data_type in DATA_TYPES:
    
    logger.info(f"DATA TYPE: {data_type.upper()}")
    
    # Construct paths dynamically for this data_type
    if PICKING_METHOD == 'ar_pick':
        picks_dir = PROJECT_ROOT / 'data' / 'processed' / '03a_phase_identification_ar_pick' / data_type
        baseline_dir = PROJECT_ROOT / 'data' / 'processed' / '04a_moment_scaling_spatial' / 'ar_pick' / data_type
    else:
        picks_dir = PROJECT_ROOT / 'data' / 'processed' / '03b_phase_identification_phasenet' / data_type
        baseline_dir = PROJECT_ROOT / 'data' / 'processed' / '04a_moment_scaling_spatial' / 'phasenet' / data_type
    
    # Load data for this data type
    signals_file = picks_dir / f'signals_{data_type}_dict.pkl'
    with open(signals_file, 'rb') as f:
        signals_dict_current = pickle.load(f)
    
    df_full_file = picks_dir / f'df_full_{data_type}_{PICKING_METHOD}.parquet'
    df_full_current = pd.read_parquet(df_full_file)
    
    logger.info(f"Loaded data: {len(signals_dict_current)} stations, {len(df_full_current)} records")
    
    # ==============================================================================
    # Loop over coda methods
    # ==============================================================================
    
    for coda_method in CODA_METHODS:
        
        logger.info(f"CODA METHOD: {coda_method}")
        
        # Load baseline results for comparison
        baseline_file = baseline_dir / coda_method / 'ensemble_spatial_summary.parquet'
        
        if not baseline_file.exists():
            logger.warning(f"Baseline not found: {baseline_file}")
            logger.warning(f"Skipping {data_type}/{coda_method}")
            continue
        
        df_baseline = pd.read_parquet(baseline_file)
        
        # Extract baseline ζ(q) for each window
        baseline_zeta = {}
        for window in ['pre_event', 'p_wave', 's_wave', 'coda']:
            window_data = df_baseline[df_baseline['window'] == window]
            if len(window_data) > 0:
                baseline_zeta[window] = window_data['zeta'].values
            else:
                baseline_zeta[window] = None
        
        # ==============================================================================
        # Loop over perturbation scenarios
        # ==============================================================================
        
        for scenario_name, scenario_params in PERTURBATION_SCENARIOS.items():
            
            current_run += 1
            logger.info(f"[{current_run}/{total_runs}] Scenario: {scenario_name}")
            
            # Apply perturbation
            if scenario_params['type'] == 'gaussian':
                df_perturbed = perturb_picks_gaussian(
                    df_full_current,
                    noise_std=scenario_params['std'],
                    sampling_rate=SAMPLING_RATE,
                    random_state=42
                )
                logger.info(f"  Applied Gaussian noise: σ={scenario_params['std']}s")
                
            elif scenario_params['type'] == 'bias':
                df_perturbed = perturb_picks_bias(
                    df_full_current,
                    bias_seconds=scenario_params['bias'],
                    sampling_rate=SAMPLING_RATE
                )
                logger.info(f"  Applied systematic bias: {scenario_params['bias']:+.1f}s")
            
            # Regenerate windowed signals with perturbed picks
            try:
                windowed_perturbed = segment_all_signals(
                    signals_dict_current,
                    df_perturbed,
                    coda_method=coda_method
                )
                logger.info(f"  Regenerated windows: {len(windowed_perturbed)} stations")
            except Exception as e:
                logger.error(f"  Failed to regenerate windows: {e}")
                continue
            
            # Compute moment scaling on perturbed windows
            try:
                results_perturbed = analyze_all_windows(
                    windowed_perturbed,
                    signal_field=SIGNAL_COLUMN_MAP[data_type],
                    tau_min=TAU_MIN,
                    n_tau=None,
                    q_values=Q_VALUES,
                    sampling_rate=SAMPLING_RATE,
                    fit_range=None
                )
                logger.info(f"  Computed moment scaling")
            except Exception as e:
                logger.error(f"  Failed to compute moment scaling: {e}")
                continue
            
            # Compare with baseline and compute metrics
            for window in ['pre_event', 'p_wave', 's_wave', 'coda']:
                
                if baseline_zeta[window] is None:
                    continue
                
                if window not in results_perturbed or results_perturbed[window] is None:
                    continue
                
                # Extract zeta from results
                zeta_perturbed = results_perturbed[window]['scaling']['zeta']
                zeta_baseline = baseline_zeta[window]
                
                metrics = compute_sensitivity_metrics(
                    zeta_baseline,
                    zeta_perturbed,
                    Q_VALUES
                )
                
                sensitivity_results[data_type][coda_method][scenario_name][window] = metrics
                
                logger.info(f"    {window:12s}: RMSE={metrics['rmse']:.4f}, "
                           f"MAE={metrics['mae']:.4f}, "
                           f"corr={metrics['correlation']:.3f}")
            
            # Save intermediate results
            intermediate_file = SENSITIVITY_INTERMEDIATE_DIR / f'results_{data_type}_{coda_method}_{scenario_name}.pkl'
            save_intermediate_results(
                {scenario_name: sensitivity_results[data_type][coda_method][scenario_name]},
                intermediate_file
            )
        
        # ==============================================================================
        # Monte Carlo analysis
        # ==============================================================================
        
        current_run += 1
        logger.info(f"[{current_run}/{total_runs}] Monte Carlo: {N_MONTE_CARLO} runs")
        
        # Store all Monte Carlo runs
        mc_zeta = {window: [] for window in ['pre_event', 'p_wave', 's_wave', 'coda']}
        
        for mc_run in range(N_MONTE_CARLO):
            
            if (mc_run + 1) % 10 == 0:
                logger.info(f"  Monte Carlo run {mc_run + 1}/{N_MONTE_CARLO}")
            
            # Perturb with different random seed each time
            df_mc = perturb_picks_gaussian(
                df_full_current,
                noise_std=MONTE_CARLO_STD,
                sampling_rate=SAMPLING_RATE,
                random_state=42 + mc_run
            )
            
            # Regenerate windows
            try:
                windowed_mc = segment_all_signals(
                    signals_dict_current,
                    df_mc,
                    coda_method=coda_method
                )
            except Exception as e:
                logger.warning(f"  MC run {mc_run} failed at windowing: {e}")
                continue
            
            # Compute moment scaling
            try:
                results_mc = analyze_all_windows(
                    windowed_mc,
                    signal_field=SIGNAL_COLUMN_MAP[data_type],
                    tau_min=TAU_MIN,
                    n_tau=None,
                    q_values=Q_VALUES,
                    sampling_rate=SAMPLING_RATE,
                    fit_range=None
                )
            except Exception as e:
                logger.warning(f"  MC run {mc_run} failed at analysis: {e}")
                continue
            
            # Store ζ(q) for each window
            for window in ['pre_event', 'p_wave', 's_wave', 'coda']:
                if window in results_mc and results_mc[window] is not None:
                    mc_zeta[window].append(results_mc[window]['scaling']['zeta'])
            
            # Save intermediate every 20 runs
            if (mc_run + 1) % 20 == 0:
                mc_intermediate = {
                    window: compute_monte_carlo_statistics(runs, Q_VALUES) 
                    for window, runs in mc_zeta.items() if len(runs) > 0
                }
                intermediate_file = SENSITIVITY_INTERMEDIATE_DIR / f'mc_{data_type}_{coda_method}_run{mc_run+1}.pkl'
                save_intermediate_results(mc_intermediate, intermediate_file)
        
        # Compute Monte Carlo statistics
        logger.info(f"  Computing Monte Carlo statistics...")
        
        for window in ['pre_event', 'p_wave', 's_wave', 'coda']:
            
            if len(mc_zeta[window]) == 0:
                continue
            
            # Compute statistics
            mc_stats = compute_monte_carlo_statistics(mc_zeta[window], Q_VALUES)
            
            # Compare mean with baseline
            if baseline_zeta[window] is not None:
                metrics = compute_sensitivity_metrics(
                    baseline_zeta[window],
                    mc_stats['mean'],
                    Q_VALUES
                )
                
                # Store both statistics and metrics
                sensitivity_results[data_type][coda_method]['monte_carlo'][window] = {
                    'statistics': mc_stats,
                    'metrics': metrics,
                    'n_successful_runs': len(mc_zeta[window])
                }
                
                logger.info(f"    {window:12s}: {len(mc_zeta[window])} successful runs, "
                           f"RMSE={metrics['rmse']:.4f}")
        
        # Save intermediate after completing method
        method_file = SENSITIVITY_INTERMEDIATE_DIR / f'complete_{data_type}_{coda_method}.pkl'
        save_intermediate_results(
            sensitivity_results[data_type][coda_method],
            method_file
        )

# ==============================================================================
# SAVE FINAL RESULTS
# ==============================================================================

logger.info("SAVING FINAL RESULTS")

# Save complete results dictionary
results_file = SENSITIVITY_OUTPUT_DIR / f'sensitivity_results_{PICKING_METHOD}_complete.pkl'
with open(results_file, 'wb') as f:
    pickle.dump(sensitivity_results, f)
logger.info(f"Saved complete results: {results_file}")

# Create and save summary table
df_summary = create_sensitivity_summary(
    sensitivity_results,
    save_path=SENSITIVITY_OUTPUT_DIR / f'sensitivity_summary_{PICKING_METHOD}.csv'
)
logger.info(f"Saved summary table: {len(df_summary)} rows")

logger.info("SENSITIVITY ANALYSIS COMPLETE")

INFO | STARTING SENSITIVITY ANALYSIS
INFO | Total configurations to test: 72
INFO | Data types: ['acceleration', 'velocity', 'displacement']
INFO | Coda methods: ['rautian', 'arias', 'envelope', 'median']
INFO | Scenarios: ['noise_small', 'noise_medium', 'noise_large', 'bias_early', 'bias_late', 'monte_carlo']
INFO | DATA TYPE: ACCELERATION
INFO | Loaded data: 21 stations, 63 records
INFO | CODA METHOD: rautian
INFO | [1/72] Scenario: noise_small
INFO |   Applied Gaussian noise: σ=0.2s
INFO |   Regenerated windows: 21 stations
INFO |   Computed moment scaling
INFO |     pre_event   : RMSE=0.0492, MAE=0.0364, corr=-0.878
INFO |     p_wave      : RMSE=0.0735, MAE=0.0550, corr=0.658
INFO |     s_wave      : RMSE=0.0479, MAE=0.0417, corr=0.981
INFO |     coda        : RMSE=0.0000, MAE=0.0000, corr=1.000
INFO | [2/72] Scenario: noise_medium
INFO |   Applied Gaussian noise: σ=0.5s
INFO |   Regenerated windows: 21 stations
INFO |   Computed moment scaling
INFO |     pre_event   : RMSE=0.0814,

Note: Converted pre_p_duration to 1000 samples @ 200Hz

SIGNAL SEGMENTATION SUMMARY
Successfully segmented: 63 signals from 21 stations
Skipped (no onset data): 0
Skipped (missing values): 0
Skipped (errors): 0

Coda detection method: rautian
Working unit: samples
Sampling rate: 200 Hz
Pre-event strategy: 1000

Pre-event window durations:
  Target: 1000 samples (5.00s)
  Actual mean: 5.00s (1000 samp)
  Actual range: [5.00, 5.00]s
ENSEMBLE SPATIAL SCALING ANALYSIS
tau_min: 0.010 s (fixed for all windows)
tau_max_fraction: None (use full window duration)
q_values: 20 values from 0.25 to 5.00
sampling_rate: 200.0 Hz

Processing window: PRE_EVENT
--------------------------------------------------------------------------------
Ensemble size: 63 signals
Tau range: [0.0100, 5.0000] s
Number of tau points: 47
Mean ζ(q): 0.0053 ± 0.0198
Mean R²: 0.0032
ζ(q=1): -0.0131
ζ(q=2): -0.0147

Processing window: P_WAVE
--------------------------------------------------------------------------------
Ens

INFO |   Computed moment scaling
INFO |     pre_event   : RMSE=0.0539, MAE=0.0416, corr=-0.954
INFO |     p_wave      : RMSE=0.2649, MAE=0.2159, corr=-0.674
INFO |     s_wave      : RMSE=0.0618, MAE=0.0521, corr=0.999
INFO |     coda        : RMSE=0.0000, MAE=0.0000, corr=1.000
INFO | [4/72] Scenario: bias_early
INFO |   Applied systematic bias: -0.5s
INFO |   Regenerated windows: 21 stations
INFO |   Computed moment scaling
INFO |     pre_event   : RMSE=0.1007, MAE=0.0896, corr=0.994
INFO |     p_wave      : RMSE=0.0214, MAE=0.0195, corr=0.405
INFO |     s_wave      : RMSE=0.0938, MAE=0.0733, corr=0.947
INFO |     coda        : RMSE=0.0000, MAE=0.0000, corr=1.000
INFO | [5/72] Scenario: bias_late
INFO |   Applied systematic bias: +0.5s
INFO |   Regenerated windows: 21 stations
INFO |   Computed moment scaling
INFO |     pre_event   : RMSE=0.1869, MAE=0.1461, corr=-0.989
INFO |     p_wave      : RMSE=0.1237, MAE=0.1050, corr=-0.759
INFO |     s_wave      : RMSE=0.3321, MAE=0.2774, corr

Ensemble size: 63 signals
Tau range: [0.0100, 0.5900] s
Number of tau points: 29
Mean ζ(q): 0.2941 ± 0.1382
Mean R²: 0.2680
ζ(q=1): 0.1409
ζ(q=2): 0.2445

Processing window: CODA
--------------------------------------------------------------------------------
Ensemble size: 63 signals
Tau range: [0.0100, 43.6050] s
Number of tau points: 66
Mean ζ(q): -0.0920 ± 0.0684
Mean R²: 0.0433
ζ(q=1): -0.0084
ζ(q=2): -0.0623

ANALYSIS COMPLETE
Note: Converted pre_p_duration to 1000 samples @ 200Hz

SIGNAL SEGMENTATION SUMMARY
Successfully segmented: 63 signals from 21 stations
Skipped (no onset data): 0
Skipped (missing values): 0
Skipped (errors): 0

Coda detection method: rautian
Working unit: samples
Sampling rate: 200 Hz
Pre-event strategy: 1000

Pre-event window durations:
  Target: 1000 samples (5.00s)
  Actual mean: 5.00s (1000 samp)
  Actual range: [5.00, 5.00]s
ENSEMBLE SPATIAL SCALING ANALYSIS
tau_min: 0.010 s (fixed for all windows)
tau_max_fraction: None (use full window duration)
q_v

INFO |   Monte Carlo run 10/100


...


SIGNAL SEGMENTATION SUMMARY
Successfully segmented: 51 signals from 17 stations
Skipped (no onset data): 0
Skipped (missing values): 0
Skipped (errors): 0

Coda detection method: median
Working unit: samples
Sampling rate: 200 Hz
Pre-event strategy: 1000

Pre-event window durations:
  Target: 1000 samples (5.00s)
  Actual mean: 5.00s (1000 samp)
  Actual range: [5.00, 5.00]s
ENSEMBLE SPATIAL SCALING ANALYSIS
tau_min: 0.010 s (fixed for all windows)
tau_max_fraction: None (use full window duration)
q_values: 20 values from 0.25 to 5.00
sampling_rate: 200.0 Hz

Processing window: PRE_EVENT
--------------------------------------------------------------------------------
Ensemble size: 51 signals
Tau range: [0.0100, 5.0000] s
Number of tau points: 47
Mean ζ(q): 0.9337 ± 0.6891
Mean R²: 0.8283
ζ(q=1): 0.6852
ζ(q=2): 1.5275

Processing window: P_WAVE
--------------------------------------------------------------------------------
Ensemble size: 51 signals
Tau range: [0.0100, 2.0650] s
Numbe

INFO |   Monte Carlo run 100/100


Ensemble size: 51 signals
Tau range: [0.0100, 2.2650] s
Number of tau points: 41
Mean ζ(q): 1.5437 ± 0.8472
Mean R²: 0.9585
ζ(q=1): 0.8881
ζ(q=2): 1.9227

Processing window: S_WAVE
--------------------------------------------------------------------------------
Ensemble size: 51 signals
Tau range: [0.0100, 8.0700] s
Number of tau points: 52
Mean ζ(q): 0.9186 ± 0.3803
Mean R²: 0.6758
ζ(q=1): 0.4834
ζ(q=2): 0.9351

Processing window: CODA
--------------------------------------------------------------------------------
Ensemble size: 51 signals
Tau range: [0.0100, 35.3550] s
Number of tau points: 64
Mean ζ(q): 0.3893 ± 0.2950
Mean R²: 0.3802
ζ(q=1): 0.2187
ζ(q=2): 0.4845

ANALYSIS COMPLETE
Note: Converted pre_p_duration to 1000 samples @ 200Hz

SIGNAL SEGMENTATION SUMMARY
Successfully segmented: 51 signals from 17 stations
Skipped (no onset data): 0
Skipped (missing values): 0
Skipped (errors): 0

Coda detection method: median
Working unit: samples
Sampling rate: 200 Hz
Pre-event strategy

/Users/giulianaparadiso/Desktop/PoliTo/Tesi/tesi-seismic-analysis/src/analysis/sensitivity.py:254: RuntimeWarning: Mean of empty slice
  'mean': np.nanmean(zeta_stack, axis=0),
/Library/Frameworks/Python.framework/Versions/3.12/lib/python3.12/site-packages/numpy/lib/_nanfunctions_impl.py:2053: RuntimeWarning: Degrees of freedom <= 0 for slice.
  var = nanvar(a, axis=axis, dtype=dtype, out=out, ddof=ddof,
/Library/Frameworks/Python.framework/Versions/3.12/lib/python3.12/site-packages/numpy/lib/_nanfunctions_impl.py:1650: RuntimeWarning: All-NaN slice encountered
  return fnb._ureduce(a,
/Users/giulianaparadiso/Desktop/PoliTo/Tesi/tesi-seismic-analysis/src/analysis/sensitivity.py:258: RuntimeWarning: All-NaN slice encountered
  'median': np.nanmedian(zeta_stack, axis=0)
INFO |   Computing Monte Carlo statistics...
INFO |     pre_event   : 100 successful runs, RMSE=0.1611
INFO |     p_wave      : 100 successful runs, RMSE=0.0640
INFO |     s_wave      : 100 successful runs, RMSE=0.3253
IN

Ensemble size: 51 signals
Tau range: [0.0100, 7.7250] s
Number of tau points: 51
Mean ζ(q): 0.7748 ± 0.3722
Mean R²: 0.6050
ζ(q=1): 0.4017
ζ(q=2): 0.7888

Processing window: CODA
--------------------------------------------------------------------------------
Ensemble size: 51 signals
Tau range: [0.0100, 35.3550] s
Number of tau points: 64
Mean ζ(q): 0.3893 ± 0.2950
Mean R²: 0.3802
ζ(q=1): 0.2187
ζ(q=2): 0.4845

ANALYSIS COMPLETE
